## 📦 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Dense, LSTM, Dropout, BatchNormalization, Input, Embedding, Concatenate, Reshape
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import Adam
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Set visualization style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print(f"✅ TensorFlow Version: {tf.__version__}")
print(f"✅ GPU Available: {tf.config.list_physical_devices('GPU')}")
print("✅ Libraries imported successfully!")

## 📂 2. Load Preprocessed Data from Module 2

In [ ]:
# Load preprocessed data
X_train = np.load('/content/X_train.npy')
X_test = np.load('/content/X_test.npy')
y_train_binary = np.load('/content/y_train_binary.npy')
y_test_binary = np.load('/content/y_test_binary.npy')
y_train_multi = np.load('/content/y_train_multi.npy')
y_test_multi = np.load('/content/y_test_multi.npy')

# Load feature names
with open('/content/feature_names.txt', 'r') as f:
    feature_names = [line.strip() for line in f.readlines()]

print("✅ Data Loaded Successfully!")
print("="*70)
print(f"📊 Feature Matrix:")
print(f"  X_train shape: {X_train.shape}")
print(f"  X_test shape:  {X_test.shape}")
print(f"  Total features: {X_train.shape[1]}")

print(f"\n🎯 Binary Classification Target:")
print(f"  y_train shape: {y_train_binary.shape}")
print(f"  y_test shape:  {y_test_binary.shape}")
print(f"  Classes: {np.unique(y_train_binary)} (0=Normal, 1=Attack)")

print(f"\n🎯 Multi-Class Target:")
print(f"  y_train shape: {y_train_multi.shape}")
print(f"  y_test shape:  {y_test_multi.shape}")
print(f"  Classes: {np.unique(y_train_multi)} (DoS, Normal, Other, Probe, R2L, U2R)")

## 🔄 3. Reshape Data for LSTM Input

LSTM expects 3D input: `(samples, timesteps, features)`

We'll treat each connection as a sequence with timesteps=1

In [ ]:
# Reshape for LSTM: (samples, timesteps, features)
# We'll use timesteps=1 since each sample is a single network connection
X_train_lstm = X_train.reshape(X_train.shape[0], 1, X_train.shape[1])
X_test_lstm = X_test.reshape(X_test.shape[0], 1, X_test.shape[1])

print("🔄 Data Reshaped for LSTM:")
print("="*70)
print(f"  X_train_lstm shape: {X_train_lstm.shape} (samples, timesteps, features)")
print(f"  X_test_lstm shape:  {X_test_lstm.shape}")
print("\n✅ Ready for LSTM model!")

## 🏗️ 4. Build LSTM Model for Binary Classification

**Architecture:**
- LSTM Layer 1: 128 units with return sequences
- Dropout: 0.3
- LSTM Layer 2: 64 units
- Dropout: 0.3
- Dense Layer 1: 32 units (ReLU)
- Dropout: 0.2
- Output Layer: 1 unit (Sigmoid) for binary classification

In [ ]:
def build_lstm_binary_model(input_shape):
    """
    Build LSTM model for Binary Classification (Normal vs Attack)
    """
    model = Sequential([
        # First LSTM layer with return sequences
        LSTM(128, return_sequences=True, input_shape=input_shape, name='lstm_1'),
        BatchNormalization(name='batch_norm_1'),
        Dropout(0.3, name='dropout_1'),
        
        # Second LSTM layer
        LSTM(64, return_sequences=False, name='lstm_2'),
        BatchNormalization(name='batch_norm_2'),
        Dropout(0.3, name='dropout_2'),
        
        # Dense layers
        Dense(32, activation='relu', name='dense_1'),
        Dropout(0.2, name='dropout_3'),
        
        # Output layer for binary classification
        Dense(1, activation='sigmoid', name='output')
    ], name='LSTM_Binary_Classifier')
    
    return model

# Build the model
input_shape = (X_train_lstm.shape[1], X_train_lstm.shape[2])  # (timesteps, features)
model_binary = build_lstm_binary_model(input_shape)

# Compile the model
model_binary.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.Precision(), tf.keras.metrics.Recall()]
)

print("🏗️ LSTM Binary Classification Model Built!")
print("="*70)
model_binary.summary()

# Count total parameters
total_params = model_binary.count_params()
print(f"\n📊 Total Parameters: {total_params:,}")

## 🎯 5. Train Binary Classification Model

In [ ]:
# Define callbacks
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=0.00001,
    verbose=1
)

model_checkpoint = ModelCheckpoint(
    '/content/best_model_binary.h5',
    monitor='val_accuracy',
    save_best_only=True,
    verbose=1
)

print("🚀 Training Binary Classification Model...")
print("="*70)

# Train the model
history_binary = model_binary.fit(
    X_train_lstm, y_train_binary,
    validation_split=0.2,
    epochs=20,
    batch_size=128,
    callbacks=[early_stopping, reduce_lr, model_checkpoint],
    verbose=1
)

print("\n✅ Training Completed!")

## 📊 6. Visualize Training History (Binary Model)

In [ ]:
# Plot training history
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('Binary Classification Model Training History', fontsize=16, fontweight='bold')

# Accuracy
axes[0, 0].plot(history_binary.history['accuracy'], label='Train Accuracy', linewidth=2)
axes[0, 0].plot(history_binary.history['val_accuracy'], label='Val Accuracy', linewidth=2)
axes[0, 0].set_title('Model Accuracy')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Accuracy')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Loss
axes[0, 1].plot(history_binary.history['loss'], label='Train Loss', linewidth=2)
axes[0, 1].plot(history_binary.history['val_loss'], label='Val Loss', linewidth=2)
axes[0, 1].set_title('Model Loss')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Loss')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Precision
axes[1, 0].plot(history_binary.history['precision'], label='Train Precision', linewidth=2)
axes[1, 0].plot(history_binary.history['val_precision'], label='Val Precision', linewidth=2)
axes[1, 0].set_title('Model Precision')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Precision')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Recall
axes[1, 1].plot(history_binary.history['recall'], label='Train Recall', linewidth=2)
axes[1, 1].plot(history_binary.history['val_recall'], label='Val Recall', linewidth=2)
axes[1, 1].set_title('Model Recall')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Recall')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("📈 Training metrics visualized!")

## 🧪 7. Evaluate Binary Model on Test Set

In [ ]:
# Make predictions
print("🔍 Evaluating on Test Set...")
print("="*70)

y_pred_binary_prob = model_binary.predict(X_test_lstm)
y_pred_binary = (y_pred_binary_prob > 0.5).astype(int).flatten()

# Calculate metrics
accuracy = accuracy_score(y_test_binary, y_pred_binary)
precision = precision_score(y_test_binary, y_pred_binary)
recall = recall_score(y_test_binary, y_pred_binary)
f1 = f1_score(y_test_binary, y_pred_binary)

print("\n📊 Binary Classification Results:")
print("="*70)
print(f"  Accuracy:  {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"  Precision: {precision:.4f}")
print(f"  Recall:    {recall:.4f}")
print(f"  F1-Score:  {f1:.4f}")

print("\n📋 Classification Report:")
print(classification_report(y_test_binary, y_pred_binary, target_names=['Normal', 'Attack']))

## 🎨 8. Confusion Matrix Visualization (Binary)

In [ ]:
# Confusion Matrix
cm_binary = confusion_matrix(y_test_binary, y_pred_binary)

plt.figure(figsize=(8, 6))
sns.heatmap(cm_binary, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Normal', 'Attack'],
            yticklabels=['Normal', 'Attack'],
            cbar_kws={'label': 'Count'})
plt.title('Confusion Matrix - Binary Classification', fontsize=14, fontweight='bold')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.tight_layout()
plt.show()

# Calculate metrics from confusion matrix
tn, fp, fn, tp = cm_binary.ravel()
print("\n📊 Confusion Matrix Breakdown:")
print("="*70)
print(f"  True Negatives (TN):  {tn:,} (Normal correctly classified)")
print(f"  False Positives (FP): {fp:,} (Normal misclassified as Attack)")
print(f"  False Negatives (FN): {fn:,} (Attack misclassified as Normal)")
print(f"  True Positives (TP):  {tp:,} (Attack correctly classified)")

## 🏗️ 9. Build LSTM Model for Multi-Class Classification

**Architecture for 6 Attack Categories:**
- DoS, Normal, Other, Probe, R2L, U2R

In [ ]:
def build_lstm_multiclass_model(input_shape, num_classes=6):
    """
    Build LSTM model for Multi-Class Classification
    """
    model = Sequential([
        # First LSTM layer
        LSTM(128, return_sequences=True, input_shape=input_shape, name='lstm_1'),
        BatchNormalization(name='batch_norm_1'),
        Dropout(0.3, name='dropout_1'),
        
        # Second LSTM layer
        LSTM(64, return_sequences=False, name='lstm_2'),
        BatchNormalization(name='batch_norm_2'),
        Dropout(0.3, name='dropout_2'),
        
        # Dense layers
        Dense(64, activation='relu', name='dense_1'),
        Dropout(0.2, name='dropout_3'),
        
        Dense(32, activation='relu', name='dense_2'),
        Dropout(0.2, name='dropout_4'),
        
        # Output layer for multi-class classification
        Dense(num_classes, activation='softmax', name='output')
    ], name='LSTM_MultiClass_Classifier')
    
    return model

# Build the model
num_classes = len(np.unique(y_train_multi))
model_multi = build_lstm_multiclass_model(input_shape, num_classes)

# Compile the model
model_multi.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("🏗️ LSTM Multi-Class Classification Model Built!")
print("="*70)
model_multi.summary()

# Count total parameters
total_params = model_multi.count_params()
print(f"\n📊 Total Parameters: {total_params:,}")

## 🎯 10. Train Multi-Class Classification Model

In [ ]:
# Define callbacks
early_stopping_multi = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

reduce_lr_multi = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=0.00001,
    verbose=1
)

model_checkpoint_multi = ModelCheckpoint(
    '/content/best_model_multiclass.h5',
    monitor='val_accuracy',
    save_best_only=True,
    verbose=1
)

print("🚀 Training Multi-Class Classification Model...")
print("="*70)

# Train the model
history_multi = model_multi.fit(
    X_train_lstm, y_train_multi,
    validation_split=0.2,
    epochs=20,
    batch_size=128,
    callbacks=[early_stopping_multi, reduce_lr_multi, model_checkpoint_multi],
    verbose=1
)

print("\n✅ Training Completed!")

## 📊 11. Visualize Training History (Multi-Class Model)

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle('Multi-Class Classification Model Training History', fontsize=16, fontweight='bold')

# Accuracy
axes[0].plot(history_multi.history['accuracy'], label='Train Accuracy', linewidth=2)
axes[0].plot(history_multi.history['val_accuracy'], label='Val Accuracy', linewidth=2)
axes[0].set_title('Model Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Loss
axes[1].plot(history_multi.history['loss'], label='Train Loss', linewidth=2)
axes[1].plot(history_multi.history['val_loss'], label='Val Loss', linewidth=2)
axes[1].set_title('Model Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("📈 Training metrics visualized!")

## 🧪 12. Evaluate Multi-Class Model on Test Set

In [ ]:
# Make predictions
print("🔍 Evaluating Multi-Class Model on Test Set...")
print("="*70)

y_pred_multi_prob = model_multi.predict(X_test_lstm)
y_pred_multi = np.argmax(y_pred_multi_prob, axis=1)

# Calculate metrics
accuracy_multi = accuracy_score(y_test_multi, y_pred_multi)
precision_multi = precision_score(y_test_multi, y_pred_multi, average='weighted')
recall_multi = recall_score(y_test_multi, y_pred_multi, average='weighted')
f1_multi = f1_score(y_test_multi, y_pred_multi, average='weighted')

print("\n📊 Multi-Class Classification Results:")
print("="*70)
print(f"  Accuracy:  {accuracy_multi:.4f} ({accuracy_multi*100:.2f}%)")
print(f"  Precision: {precision_multi:.4f} (weighted)")
print(f"  Recall:    {recall_multi:.4f} (weighted)")
print(f"  F1-Score:  {f1_multi:.4f} (weighted)")

# Class names
class_names = ['DoS', 'Normal', 'Other', 'Probe', 'R2L', 'U2R']

print("\n📋 Classification Report:")
print(classification_report(y_test_multi, y_pred_multi, target_names=class_names))

## 🎨 13. Confusion Matrix Visualization (Multi-Class)

In [ ]:
# Confusion Matrix
cm_multi = confusion_matrix(y_test_multi, y_pred_multi)

plt.figure(figsize=(10, 8))
sns.heatmap(cm_multi, annot=True, fmt='d', cmap='YlOrRd', 
            xticklabels=class_names,
            yticklabels=class_names,
            cbar_kws={'label': 'Count'})
plt.title('Confusion Matrix - Multi-Class Classification', fontsize=14, fontweight='bold')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.tight_layout()
plt.show()

print("✅ Confusion matrix visualized!")

## 📋 14. Module 3 Summary Report

In [ ]:
print("\n" + "="*70)
print("📋 MODULE 3: MODEL BUILDING - SUMMARY REPORT")
print("="*70)

print("\n🧠 DEEP LEARNING ARCHITECTURE:")
print("   Model Type: LSTM (Long Short-Term Memory)")
print("   Framework: TensorFlow/Keras")
print(f"   Total Parameters (Binary): {model_binary.count_params():,}")
print(f"   Total Parameters (Multi-Class): {model_multi.count_params():,}")

print("\n📊 BINARY CLASSIFICATION RESULTS:")
print("="*70)
print(f"   Task: Normal vs Attack Detection")
print(f"   ✓ Accuracy:  {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"   ✓ Precision: {precision:.4f}")
print(f"   ✓ Recall:    {recall:.4f}")
print(f"   ✓ F1-Score:  {f1:.4f}")

print("\n📊 MULTI-CLASS CLASSIFICATION RESULTS:")
print("="*70)
print(f"   Task: 6-Class Attack Categorization")
print(f"   Classes: DoS, Normal, Other, Probe, R2L, U2R")
print(f"   ✓ Accuracy:  {accuracy_multi:.4f} ({accuracy_multi*100:.2f}%)")
print(f"   ✓ Precision: {precision_multi:.4f} (weighted)")
print(f"   ✓ Recall:    {recall_multi:.4f} (weighted)")
print(f"   ✓ F1-Score:  {f1_multi:.4f} (weighted)")

print("\n🎯 MODEL FEATURES:")
print("   ✓ Two-layer LSTM architecture")
print("   ✓ Batch Normalization for stable training")
print("   ✓ Dropout layers (0.2-0.3) for regularization")
print("   ✓ Early Stopping to prevent overfitting")
print("   ✓ Learning Rate Reduction on plateau")
print("   ✓ Model Checkpointing for best weights")

print("\n💾 SAVED MODELS:")
print("   ✅ /content/best_model_binary.h5")
print("   ✅ /content/best_model_multiclass.h5")

print("\n" + "="*70)
print("✅ MODULE 3 COMPLETED SUCCESSFULLY!")
print("🚀 Ready for Module 4: Evaluation & Performance Analysis")
print("="*70)

## 💾 15. Save Training History for Module 4

In [ ]:
# Save predictions and probabilities for Module 4 analysis
np.save('/content/y_pred_binary.npy', y_pred_binary)
np.save('/content/y_pred_binary_prob.npy', y_pred_binary_prob)
np.save('/content/y_pred_multi.npy', y_pred_multi)
np.save('/content/y_pred_multi_prob.npy', y_pred_multi_prob)

# Save training history
import pickle
with open('/content/history_binary.pkl', 'wb') as f:
    pickle.dump(history_binary.history, f)
    
with open('/content/history_multi.pkl', 'wb') as f:
    pickle.dump(history_multi.history, f)

print("\n💾 Saved for Module 4:")
print("="*70)
print("✅ y_pred_binary.npy")
print("✅ y_pred_binary_prob.npy")
print("✅ y_pred_multi.npy")
print("✅ y_pred_multi_prob.npy")
print("✅ history_binary.pkl")
print("✅ history_multi.pkl")
print("\n🚀 Ready for Module 4: Evaluation!")